In [ ]:
import sys
import xarray as xr
import numpy as np
import pandas as pd
import math
import glob
import geopandas as gpd

import cartopy
import matplotlib.pyplot as plt
import cmocean.cm as cmo
from matplotlib.gridspec import GridSpec
from matplotlib.colorbar import Colorbar # different way to handle colorbar

# cartopy
import cartopy.crs as ccrs
from cartopy.mpl.geoaxes import GeoAxes
import cartopy.feature as cfeature
import dask

# import personal modules
# Path to modules
sys.path.append('../modules')
# Import my modules
import global_vars
from utils import roundPartial
from plotter import draw_basemap, plot_terrain, plot_sensitivity_cbar
from trajectory import calculate_backward_trajectory
import customcmaps as ccmaps

dask.config.set(**{'array.slicing.split_large_chunks': True})

In [ ]:
path_to_data = global_vars.path_to_data

In [ ]:
## what was the starting pressure at the site for each of these storms?


In [ ]:
HUC8_lst = [14050001, 13010001, 10190002, 11020001]
HUC8_lbl = ['Upper Yampa', 'Rio Grande Headwaters', 'Upper South Platte', 'Arkansas Headwaters']

ds_lst = []
## loop through all HUC8s
for i, HUC8_ID in enumerate(HUC8_lst):
    fname = path_to_data+'preprocessed/ERA5_trajectories/sensitivity_tests/PRISM_HUC8_{0}.nc'.format(HUC8_ID)
    ds = xr.open_dataset(fname)
    ds = ds.assign(grid=['center', 'east', 'north', 'south', 'west'])
    ds_lst.append(ds)

ds = xr.concat(ds_lst, pd.Index(HUC8_lst, name="HUC8"))
ds

In [ ]:
## Load subbasin watershed file
fp = path_to_data + 'downloads/CO_HUC8/wbdhu8.shp'
polys = gpd.read_file(fp, crs="epsg:4326") # have to manually set the projection


idx = (polys.HUC8 == str(HUC8_lst[0])) | (polys.HUC8 == str(HUC8_lst[1])) | (polys.HUC8 == str(HUC8_lst[2])) | (polys.HUC8 == str(HUC8_lst[3]))
polys =  polys[idx]
polys

In [ ]:
storm = 'mar2003'
if storm == 'mar2003':
    tmp = ds.sel(start_date=slice('2003-03-18 12', '2003-03-19 06'))
    yr = '2003'
    mon = '03'
elif storm == 'mar2019':
    tmp = ds.sel(start_date=slice('2019-03-13 12', '2019-03-14  06'))
    yr = '2019'
    mon = '03'
elif storm == 'jan2017':
    tmp = ds.sel(start_date=slice('2017-01-09 12', '2019-01-10  06'))
    yr = '2017'
    mon = '01'
elif storm == 'sept2013':
    tmp = ds.sel(start_date=slice('2013-09-10 12', '2019-09-11  06'))
    yr = '2013'
    mon = '09'
tmp

In [ ]:
## load geopotential height data
fname = path_to_data + 'downloads/ERA5/{0}{1}_z_prs.nc'.format(yr, mon)
gph = xr.open_dataset(fname)
gph

In [ ]:
gph.isel(valid_time=0, pressure_level=3).z.values/10

In [ ]:
## create tick labels 'YYYY-MM-DD HH'
t_lst = []
for m, start_date_val in enumerate(tmp.start_date.values):
    ts = pd.to_datetime(str(start_date_val)) 
    t = ts.strftime('%Y-%m-%d %H UTC')
    t_lst.append(t)
# ["{:.0%}".format(i) for i in cbax.get_ticks()]
t_lst

In [ ]:
letter_lst = list(map(chr, range(97, 123)))
title_list = []
for i, col in enumerate(np.arange(0,16, 4)):
    for j, row in enumerate(np.arange(0,4,1)):
        print(i, j)
        titlestring = '({0})'.format(letter_lst[col+j])
        print(titlestring)

In [ ]:
ext = [-130., -90., 20., 60.] # original submitted
ext = [-125., -95., 25., 55.]

fmt = 'png'
nrows = 6
ncols = 4
colors = ['#ffffcc', '#a1dab4', '#41b6c4', '#225ea8']

# Set up projection
datacrs = ccrs.PlateCarree()  ## the projection the data is in
mapcrs = ccrs.PlateCarree() ## the projection you want your map displayed in

# Set tick/grid locations
tx = 10
ty = 5
dx = np.arange(ext[0],ext[1]+tx,tx)
dy = np.arange(ext[2],ext[3]+ty,ty)

fig = plt.figure(figsize=(12.0 ,12.0))
fig.dpi = 600
fname = '../figs/sensitivity_test_trajectory_{0}'.format(storm)

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 1, 0.05, 0.05], width_ratios = [1, 1, 1, 1], wspace=0.01, hspace=0.05)
## use gs[rows index, columns index] to access grids

left_lats = [True, False, False, False]
letter_idx = np.arange(0,16, 4)
## each 6 hours gets its own plot
for i, start_date_val in enumerate(tmp.start_date.values):
    
    for j, lev_val in enumerate(tmp.start_lev.values):
        print(i, j, lev_val)
        ax = fig.add_subplot(gs[j, i], projection=mapcrs)
        ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy,left_lats=left_lats[i], right_lats=False)
        ax.set_extent(ext, datacrs)
        ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)

        ## add geopotential height contours
        gph_data = gph.sel(valid_time=start_date_val, pressure_level=lev_val)
        clevs_hgts = np.arange(0,8000,4)
        z = gph_data.z.values / 9.80665 # convert from geopotential to geopotential height in m
        cs = ax.contour(gph.longitude.values, gph.latitude.values, z/10., transform=datacrs,
                    levels=clevs_hgts, colors='grey', linewidths=0.5)
        plt.clabel(cs, fmt='%d',fontsize=6, inline_spacing=5)  
        
        if j == 0:
            ax.set_title(t_lst[i], loc='left', fontsize=11)

        if i == 0: # add row labels to the far left plot
            ax.text(-0.20, 0.5, '{0} hPa'.format(str(int(lev_val))), va='bottom', ha='center',
                rotation='vertical', rotation_mode='anchor', fontsize=13,
                transform=ax.transAxes)
            
        ## start with start date and level
        data = tmp.sel(start_lev=lev_val, start_date=start_date_val)
        
        # need this to fix annotate transform
        transform = datacrs._as_mpl_transform(ax)
        ## Loop through grid
        for k, grid_val in enumerate(data.grid.values):
            for m, HUC8 in enumerate(data.HUC8.values):
                d = data.sel(grid=grid_val, HUC8=HUC8)
                y_lst = d.latitude.values
                x_lst = d.longitude.values
                # z_lst = d.level.values
                z_lst = d.q.values
                # cmap, norm, bnds = ccmaps.cmap('pressure')
                cf = ax.scatter(x_lst, y_lst, c=colors[m], marker='.', transform=datacrs, alpha=0.8, s=6, zorder=100)
                # cf = ax.scatter(x_lst, y_lst, c=z_lst, marker='.', transform=datacrs, alpha=0.8, s=6, zorder=100)

        ## add in four focus watersheds
        # polys.plot(ax=ax, edgecolor='white', color='None', zorder=99)
        col = letter_idx[i]
        titlestring = '({0})'.format(letter_lst[col+j])
        ax.text(0.03, 0.96, titlestring, ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)
    
        
# Add color bar
cbax = plt.subplot(gs[-1,:]) # colorbar axis
plot_sensitivity_cbar(cbax, orientation='horizontal')
kw_ticklabels = {'size': 8, 'color': 'dimgray', 'weight': 'light'}
cbax.set_xticklabels(HUC8_lbl, **kw_ticklabels)  # horizontally oriented colorbar
# cb = Colorbar(ax = cbax, mappable = cf, orientation = 'horizontal', ticklocation = 'bottom')
# cb.set_label('Specific Humidity (kg kg$^{-1}$)', fontsize=11)
# cb.ax.tick_params(labelsize=12)

fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
plt.show()
fig.clf()